# 🔑 Gerar o GOOGLE_TOKEN (autorização para o robô criar arquivos no seu Drive)

O Google não deixa a conta de serviço (o robô) **criar** arquivos no seu Drive pessoal, só atualizar os que já existem.
Com este token o robô passa a agir **em seu nome**: faz isso uma vez só.

**Antes de rodar** (no Google Cloud, projeto `supple-antenna-509519-h6`):
1. **APIs e serviços → Tela de consentimento OAuth** (ou "Google Auth Platform"): tipo **Externo**, nome `Robo METAR`, seu e-mail. Em **Público-alvo**, clique em **Publicar app** ("Em produção"). Assim o token não expira em 7 dias.
2. **Credenciais → Criar credenciais → ID do cliente OAuth → Tipo: App para computador** → Criar → **Baixar JSON**.

Depois rode as células abaixo.

In [ ]:
# 1) Envie o JSON do cliente OAuth que você baixou
!pip -q install google-auth-oauthlib
from google.colab import files
import json
up = files.upload()
cliente = json.loads(list(up.values())[0])
print('✅ Cliente carregado')

In [ ]:
# 2) Abra o link, entre com a SUA conta e autorize.
#    Se aparecer "O Google não verificou este app": Avançado -> Acessar Robo METAR.
#    No fim o navegador vai para uma página "localhost" que NÃO abre — é normal.
#    Copie o endereço inteiro dessa página (barra de endereço) e cole aqui.
from google_auth_oauthlib.flow import Flow
flow = Flow.from_client_config(cliente, scopes=['https://www.googleapis.com/auth/drive'],
                               redirect_uri='http://localhost:1')
url, _ = flow.authorization_url(access_type='offline', prompt='consent')
print(url)
resposta = input('Cole aqui o endereço da página localhost: ').strip()
import os; os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'
flow.fetch_token(authorization_response=resposta)
c = flow.credentials
token = json.dumps({'client_id': c.client_id, 'client_secret': c.client_secret,
                    'refresh_token': c.refresh_token})
print('\n✅ Copie a linha abaixo INTEIRA e cole no GitHub:')
print('   Settings -> Secrets and variables -> Actions -> New repository secret')
print('   Nome: GOOGLE_TOKEN\n')
print(token)